# Tool Loadout: Smarter Context for a Support Agent

In this notebook, we’ll explore how **Tool Loadout** helps a customer support agent choose only the tools it really needs for each request. We’ll compare two cases, one where the agent uses all available tools and another where it picks just a few relevant ones based on context.

By the end, you’ll see how this approach helps reduce latency and token usage without changing the core logic of the agent.

<img src="./images/img_01.png" alt="Diagram" width="500">

## Setting Up

Before we begin, we set up the workspace: load configuration and import what we need. We also **silence the noisy third-party system logs** (OpenAI/httpx HTTP request lines, Chroma telemetry) so the only output is our own `print` statements — which we use to trace the tool-loadout mechanism under the hood.

- **build_tool_index** (`rag.indexer`) – embeds every tool's description and builds the vector index used for semantic tool search.
- **create_agent** (`langchain.agents`) + **ChatOpenAI** – the LangChain primitives we'll use to build the agent (defined a few cells down).
- **load_dotenv** – loads environment variables from a `.env` file (API keys, model config).
- **HumanMessage** (`langchain_core.messages`) – the message type we pass into the agent.

In [9]:
import logging
import os

# Silence noisy third-party system logs (OpenAI/httpx HTTP requests, Chroma
# telemetry, etc.) so the only output we see is our own print statements.
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.disable(logging.INFO)

from config import PERSIST_DIR, LLM_MODEL, LLM_TEMPERATURE
from rag.indexer import build_tool_index
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import BaseTool
from langchain_core.messages import HumanMessage

In [10]:
load_dotenv()

True

## Loading the Tool Catalog

Our support agent can use a set of predefined tools — things like checking order details, verifying payments, or handling refunds. Each tool has a name, a description, and a function that defines what it does.

We build a vector index of all available tools (order lookup, payment verification, refunds, etc.) for semantic search. This index enables Tool Loadout to retrieve only the most relevant tools for each query.

In [11]:
# Build the tool index once. This embeds every tool description so we can
# semantically retrieve the most relevant tools for a given query.
print("Building tool index…")
vector_store, name_to_card = build_tool_index(PERSIST_DIR)
print(f"Done — indexed {len(name_to_card)} tools into Chroma.")

Building tool index…
  Using OpenAIEmbeddings (model=text-embedding-3-small)
  Reusing existing Chroma index (25 tools, persist_dir='.chroma_tools')
Done — indexed 25 tools into Chroma.


## Building the Agent

Given a list of tools, this helper builds a tool-calling support agent with `create_agent` — LangChain's recommended primitive for agents. The agent runs the standard reason → call tool → observe loop until it can answer.

The key idea: we pass in **only the tools we want this agent to have**. Tool Loadout (below) decides which tools that is.

> **Note on the model:** we deliberately run on a smaller model (`gpt-4o-mini`, set via `LLM_MODEL` in `.env`). A strong model can usually stay on track even when handed all 25 tools, so the cost of a bloated toolbox is mostly hidden. A weaker model is more easily distracted by confusable tools — which is exactly when narrowing the loadout earns its keep.

In [12]:
SYSTEM_PROMPT = (
    "You are a helpful customer support agent. Use the tools to gather facts and "
    "resolve the request precisely. Ask focused follow-ups if data is missing. "
    "Be concise and friendly."
)


def build_agent(selected_tools: list[BaseTool]):
    """Build a tool-calling support agent restricted to `selected_tools`."""
    llm = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
    return create_agent(llm, selected_tools, system_prompt=SYSTEM_PROMPT)

## Selecting Tools and Handling a Request

Three small pieces drive the demo:

- **`select_tools`** decides *which* tools the agent gets — the **loadout**. With Tool Loadout on, it semantically retrieves the top-`k` tools for the query; with it off, it returns the full catalog.
- **`service_request`** ties it together: select the loadout → build an agent with just those tools → run the query.
- **`report_run`** prints a clear view of each run: the **tool loadout** (which tools were loaded, and which the agent actually used), the **agent behavior** (the tool calls it made, in order, with arguments), and the **final answer**.

In [13]:
TOP_K = 5


def select_tools(user_query: str, tool_loadout: bool = True, k: int = TOP_K) -> list[str]:
    """Return the tool names to load for this query.

    With Tool Loadout, semantically retrieve the top-`k` tools; otherwise return
    the entire catalog. Kept quiet so it can be reused (e.g. in the intent table).
    """
    if not tool_loadout:
        return list(name_to_card.keys())

    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(user_query)
    return [doc.metadata["name"] for doc in docs if doc.metadata.get("name")]


def service_request(user_query: str, user_id: str, tool_loadout: bool = True) -> dict:
    """Resolve one support request end-to-end and return the loadout + result."""
    mode = "ON" if tool_loadout else "OFF"
    print(f"Running agent (tool loadout {mode})… ", end="", flush=True)

    tool_names = select_tools(user_query, tool_loadout)
    selected_tools = [name_to_card[name].handle for name in tool_names]

    agent = build_agent(selected_tools)
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_query)]},
        config={"run_name": "support-agent", "metadata": {"user_id": user_id}},
    )
    print("done.")
    return {"selected_tools": tool_names, "agent_result": result}


def agent_tool_calls(result: dict) -> list[tuple[str, dict]]:
    """The (tool_name, args) pairs the agent actually invoked, in call order."""
    return [
        (call["name"], call.get("args", {}))
        for message in result["messages"]
        for call in getattr(message, "tool_calls", None) or []
    ]


def agent_token_usage(result: dict) -> tuple[int, int]:
    """Total (input, output) tokens across all of the agent's LLM turns.

    This is where the loadout pays off: every loaded tool's schema is part of
    the prompt on *every* LLM turn, so a bigger toolbox means more input tokens.
    """
    in_tok = out_tok = 0
    for message in result["messages"]:
        usage = getattr(message, "usage_metadata", None) or {}
        in_tok += usage.get("input_tokens", 0)
        out_tok += usage.get("output_tokens", 0)
    return in_tok, out_tok


def report_run(title: str, payload: dict) -> None:
    """Show the tool loadout, the agent's behavior, and the token cost for one run."""
    loaded = payload["selected_tools"]
    result = payload["agent_result"]
    calls = agent_tool_calls(result)
    called_names = {name for name, _ in calls}

    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

    # 1) The loadout: which tools the agent was given (and which it used).
    print(f"\nTool loadout — {len(loaded)} tool(s) loaded into the agent:")
    for name in loaded:
        marker = "✓ called" if name in called_names else "  unused"
        print(f"  [{marker}] {name}")

    # 2) The behavior: which tools it called, in order, with arguments.
    print(f"\nAgent behavior — {len(calls)} tool call(s):")
    if calls:
        for i, (name, args) in enumerate(calls, 1):
            arg_str = ", ".join(f"{k}={v!r}" for k, v in args.items())
            print(f"  {i}. {name}({arg_str})")
    else:
        print("  (the agent answered without calling any tools)")

    # 3) The context cost: input tokens scale with how many tools were loaded.
    in_tok, out_tok = agent_token_usage(result)
    print(
        f"\nContext cost — {in_tok} input + {out_tok} output "
        f"= {in_tok + out_tok} tokens"
    )

    # 4) The final answer.
    print("\nFinal answer:")
    print(f"  {result['messages'][-1].content}")

<img src="./images/img_02.png" alt="Diagram" width="500">

## Demo Queries

Our main example is a **stolen gift card**: the customer's card `GC-111` (which really exists in the mock data, with a ₹500 balance) was compromised, and they want us to check the balance and block it. The clean resolution is two steps — `giftcard_balance` → `block_giftcard` — and those are exactly the two tools the loadout surfaces for this query.

This is a good loadout example because the full 25-tool catalog is full of *confusable* neighbours: other "blocking / security" actions (`two_factor_reset`), and a whole family of ticket tools (`create_ticket`, `create_refund_ticket`, `escalate_ticket`). Handed everything at once — especially with a smaller model — the agent is more likely to grab a wrong-but-plausible tool or make extra calls. Narrowing the loadout removes those distractors.

We also define a few more queries covering different intents — warranty, refund, loyalty, tickets, security — to show later that Tool Loadout adapts the toolset to each request.

In [14]:
# Main demo query (grounded in the mock data: gift card GC-111 exists with a ₹500 balance).
DEMO_QUERY = (
    "My gift card GC-111 was stolen. Can you check the balance and then "
    "block it so no one else can use it?"
)

# A few more intents, to show how loadout picks a different toolset per request.
DEMO_QUERIES = {
    "giftcard": DEMO_QUERY,
    "warranty": "Is my Smartwatch Z (order B2001) still under warranty, and can I extend it? My user id is user_456.",
    "refund":   "I think I was charged twice for order A1099 and I'd like a refund. My user id is user_123.",
    "loyalty":  "What's my loyalty points balance and what should I buy next? My user id is user_123.",
    "tickets":  "What open support tickets do I have, and can you escalate the refund one? My user id is user_123.",
    "security": "I'm locked out of my account and need to reset my two-factor authentication. My user id is user_123.",
}

## Understanding How Tool Loadout Works

Normally, the model has access to every tool at once. That means it reads through all descriptions before deciding which ones to use. This can take extra time and tokens.

With Tool Loadout, we make this process smarter. We use embeddings to represent each tool, then perform a similarity search based on the user query. Only the top few most relevant tools are loaded for that specific request.

It’s like showing the model only the tools it actually needs for the job instead of dumping the entire toolbox.

<img src="./images/img_03.png" alt="Diagram" width="500">

In [17]:
# Run 1: WITH Tool Loadout — only the top-5 semantically relevant tools are loaded.
payload_loadout = service_request(DEMO_QUERY, user_id="user_123", tool_loadout=True)
report_run("WITH tool loadout (top-5)", payload_loadout)

Running agent (tool loadout ON)… done.

WITH tool loadout (top-5)

Tool loadout — 5 tool(s) loaded into the agent:
  [✓ called] block_giftcard
  [✓ called] giftcard_balance
  [  unused] check_warranty
  [  unused] loyalty_points
  [  unused] two_factor_reset

Agent behavior — 2 tool call(s):
  1. giftcard_balance(code='GC-111')
  2. block_giftcard(code='GC-111')

Context cost — 770 input + 94 output = 864 tokens

Final answer:
  The gift card with code GC-111 had a balance of 500.0 INR. It has now been successfully blocked to prevent any unauthorized use. If you need further assistance, feel free to ask!


## Running Without Tool Loadout (All Tools)

For contrast, we run the same query with the **entire catalog** loaded. The agent will likely still reach the same answer — but every one of the 25 tool schemas is now part of the prompt on *each* LLM turn. Watch two numbers in the report below: the **"Tool loadout"** count (5 vs 25) and the **"Context cost"** line. Same outcome, several times the input tokens — and on a weaker model, more room to grab a confusable tool.

In [15]:
# Run 2: WITHOUT Tool Loadout — the agent gets the entire catalog and must
# read every description before deciding what to call.
payload_all = service_request(DEMO_QUERY, user_id="user_123", tool_loadout=False)
report_run("WITHOUT tool loadout (all tools)", payload_all)

Running agent (tool loadout OFF)… done.

WITHOUT tool loadout (all tools)

Tool loadout — 25 tool(s) loaded into the agent:
  [  unused] faq_search
  [  unused] crm_lookup
  [  unused] payment_history_lookup
  [  unused] order_status
  [  unused] refund_policy_check
  [  unused] create_refund_ticket
  [  unused] shipping_status
  [  unused] shipping_update_address
  [  unused] escalate_ticket
  [  unused] create_ticket
  [  unused] list_open_tickets
  [  unused] close_ticket
  [  unused] loyalty_points
  [  unused] apply_coupon
  [  unused] resend_invoice
  [  unused] verify_email
  [  unused] update_email
  [  unused] add_note_to_crm
  [  unused] recommend_product
  [  unused] check_warranty
  [  unused] extend_warranty
  [✓ called] giftcard_balance
  [✓ called] block_giftcard
  [  unused] cancel_order
  [  unused] two_factor_reset

Agent behavior — 2 tool call(s):
  1. giftcard_balance(code='GC-111')
  2. block_giftcard(code='GC-111')

Context cost — 5192 input + 77 output = 5269 tok

## Loadout Adapts to Intent

The real payoff: the *same* mechanism selects a *different* toolset for every kind of request. Below we run only the retrieval step (no LLM call) for each demo query and show which 5 tools get loaded. Notice how refund/return queries surface payment and refund tools, while warranty and loyalty queries surface entirely different ones.

In [16]:
print(f"{'Intent':<9} | Top-5 tools selected by loadout")
print("-" * 78)
for intent, query in DEMO_QUERIES.items():
    selected = select_tools(query, tool_loadout=True)
    print(f"{intent:<9} | {', '.join(selected)}")

Intent    | Top-5 tools selected by loadout
------------------------------------------------------------------------------
giftcard  | block_giftcard, giftcard_balance, check_warranty, loyalty_points, two_factor_reset
warranty  | check_warranty, extend_warranty, shipping_status, loyalty_points, giftcard_balance
refund    | resend_invoice, create_refund_ticket, payment_history_lookup, check_warranty, cancel_order
loyalty   | loyalty_points, payment_history_lookup, giftcard_balance, recommend_product, check_warranty
tickets   | create_refund_ticket, escalate_ticket, create_ticket, list_open_tickets, crm_lookup
security  | two_factor_reset, verify_email, update_email, create_ticket, crm_lookup


## What We Learned

In this notebook, we saw how Tool Loadout helps an agent focus on what matters.

Both runs reached the **same correct answer** — check the balance, block the card — but the all-tools run carried ~4–5× the input tokens, because every unused tool schema still rides along in the prompt on every turn. That gap is the cost of a bloated toolbox: more tokens, more latency, and (especially on smaller models or vaguer requests) more room to pick a confusable tool.

By narrowing context to only the most relevant tools, Tool Loadout keeps the same behavior while cutting that cost.

You've just seen Context Engineering in action 🎉